# Cross-talk research: does a target gene influence the pathway of a core (disease-causal) gene?

**Inputs: `CORE` (e.g. disease-causal gene) and `TARGET` (gene of interest).** The downstream pathway of each gene is computed **with the other gene removed**, so only routes independent of the other gene count. Output: shared nodes, *interference edges* (TARGET-side node acting on a CORE-pathway node, with residue/mechanism), endpoint agreement (synergistic / antagonistic), and a cross-talk index.

## 1. Parameters

Database locations come from the YAML config (`config.example.yaml` → copy to `target_net.yaml`); parameters here only override it.

In [ ]:
CORE = "FGFR3"        # core / disease-causal gene
TARGET = "HSP90AA1"   # gene of interest (HSP90AA2 is resolved to HSP90AA1 automatically)
DEMO = False           # False = your databases from target_net.yaml; True = offline curated demo graph
CONFIG = None          # None = find target_net.yaml in this folder or any parent (project root), else ~/.target_net.yaml; or give a path
DEPMAP_DIR = None      # overrides databases.depmap.dir in the YAML
TISSUE = None
MAX_DEPTH = MIN_SCORE = HUB_PENALTY = None   # None = values from the YAML (walk: section)
TARGET_NET_DIR = ".."
FOCUS = None           # genes to highlight: list ["GSK3B","FOXO3"], a file (symbol[,value] per line), or dict {"GSK3B": -1.8}; others drawn as small grey dots

## 2. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
sys.path.insert(0, os.path.abspath(TARGET_NET_DIR))
from targetnet.api import research_crosstalk, summarize_crosstalk, show, figure, legend
from targetnet.plots import use_cdn_renderer
use_cdn_renderer()   # plotly.js from CDN -> small .ipynb files
import pandas as pd
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
OUT_ROOT = os.path.join(TARGET_NET_DIR, "runs")   # each analysis gets runs/<YYYYMMDD_HHMMSS>_<mode>_<genes>/

### Key to the figures
Every figure carries its own caption (lines, arrowheads, colours). `legend(kind)` prints the same key as text — kinds: `flow`, `cell_map`, `crosstalk`, `volcano`.

In [ ]:
legend("crosstalk")

## 3. Run

In [ ]:
x = research_crosstalk(core=CORE, target=TARGET, config=CONFIG, demo=DEMO, depmap=os.path.expanduser(DEPMAP_DIR) if DEPMAP_DIR else None, tissue=TISSUE,
                       max_depth=MAX_DEPTH, min_score=MIN_SCORE, hub_penalty=HUB_PENALTY, out=OUT_ROOT)
print(summarize_crosstalk(x))
print("outputs:", x["run_dir"])

## 4. Interference edges (ranked by proximity)
Each row: a node reachable only from TARGET acting on a node of the CORE pathway (or on CORE itself). `rank` orders by `hops_goi_to_from` + `hops_causal_to_to` (closest first) then edge confidence. `net_effect_of_goi_on_target` = sign(TARGET→from) × edge effect, compared with CORE's own effect on that node → synergistic / antagonistic. `mechanism` distinguishes phosphorylation from chaperone/stability edges; `via_feedback_loop` flags from-nodes that are also upstream of TARGET.

In [ ]:
x["interference"][["rank","from_node","to_node","hops_goi_to_from","hops_causal_to_to","mechanism","residue","net_effect_of_goi_on_target","causal_effect_on_target","agreement","via_feedback_loop"]].head(30)

## 5. Shared nodes and endpoint agreement
Where do the two pathways converge, and do they push the endpoints (TFs / target genes) in the same direction?

In [ ]:
x["shared"].sort_values("hops_from_causal")

In [ ]:
x["endpoints"]

## 6. Views
Three lanes: CORE pathway / shared / TARGET pathway. Thick green = synergistic interference, red = antagonistic, long-dashed = chaperone edge. Click nodes and edges for details.

In [ ]:
figure(x["views"]["crosstalk"], focus=FOCUS).show()          # Plotly, inline. Full D3 view: show(x["views"]["crosstalk"])

In [ ]:
figure(x["views"]["core_flow"], focus=FOCUS).show()          # Plotly, inline. Full D3 view: show(x["views"]["core_flow"])

## 7. How to interpret
1. **Is the overlap more than chance?** Look at `TF overlap … expected … enrichment … p`. A hub TARGET (AKT1, MAPK1, HSP90) reaches a large share of all TFs, so the raw cross-talk index saturates near 1 for *any* core gene; the hypergeometric enrichment and p-value correct for that. Enrichment ≫ 1 with p < 0.05 = specific cross-talk; enrichment ≈ 1 = TARGET touches this pathway no more than any other.
2. **Where does TARGET touch the CORE pathway?** The interference table is ranked by proximity (`hops_goi_to_from` + `hops_causal_to_to`). Rank 1–10 with 0–1 hops on both sides are the direct, testable mechanisms (e.g. a kinase phosphorylating a CORE-pathway component at a named residue). Edges 3+ hops away on both sides are weak hypotheses.
3. **Direction**: `antagonistic` = TARGET pushes a CORE-pathway node the opposite way to CORE (a brake; inhibiting TARGET releases it) — most relevant for combination therapy. `synergistic` = same direction (TARGET inhibition would reinforce CORE inhibition). `low-confidence` = one of the signs is not settled (conflicting routes); do not over-read.
4. **Endpoint counts** (syn/ant/mixed/low-conf) summarise TFs and target genes; a large `mixed`/`low-conf` count means the walk is too long for sign to propagate reliably — lower `MAX_DEPTH` or raise `MIN_SCORE` rather than trusting the counts.
5. **Baseline**: run the same TARGET against 3–5 unrelated core genes; if enrichment is similar everywhere, the signal is the hub, not the pathway.
6. Interference edges with a residue are directly testable by phosphoproteomics (CORE inhibitor ± TARGET inhibitor).